# Vertical Mode

In [1]:
import pandas as pd
import osmnx as ox
from pyrosm import OSM
import numpy as np

In [2]:
# Grafo peatonal
USAR_ARCHIVO_LOCAL = True
PBF_PATH = "madrid-261025.osm.pbf"

if USAR_ARCHIVO_LOCAL:
    # Requiere: uv pip install pyrosm
    osm = OSM(PBF_PATH)
    nodes, edges = osm.get_network(network_type="walking", nodes=True)
else:
    G = ox.graph_from_place(
        "Madrid, Spain",
        network_type="walk"
    )
    nodes, edges = ox.graph_to_gdfs(G)

print(len(edges))

2594752


In [3]:
import rasterio
#RASTER_PATH = "MDT05_Madrid.tif"
RASTER_PATH = "PNOA_MDT05_ETRS89_HU30_0559_LID.tif"

raster = rasterio.open(RASTER_PATH)
print(raster.crs)

LOCAL_CS["Transverse Mercator (ETRS89-TM30)",UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


In [4]:
# El raster usa un LOCAL_CS sin código EPSG; ETRS89 / UTM 30N es EPSG:25830
edges = edges.to_crs("EPSG:25830")

In [5]:
with rasterio.open(RASTER_PATH) as src:
    coords = []
    for geom in edges.geometry:
        for t in np.linspace(0, 1, 10):
            p = geom.interpolate(t, normalized=True)
            coords.append((p.x, p.y))
    vals = np.array(list(src.sample(coords))).ravel()

vals = vals.reshape(len(edges), 10)
edges["elev_mean"] = vals.mean(axis=1)
edges["elev_min"] = vals.min(axis=1)
edges["elev_max"] = vals.max(axis=1)

MemoryError: 

In [ ]:
edges["elev_mean"] = edges["elev_mean"]
edges["elev_min"] = edges["elev_min"]
edges["elev_max"] = edges["elev_max"]

In [ ]:
edges["desnivel"] = (
    edges["elev_max"] -
    edges["elev_min"]
)

In [ ]:
edges["length_m"] = edges.geometry.length


In [ ]:
edges["pendiente_pct"] = (
    edges["desnivel"] /
    edges["length_m"]
) * 100

In [ ]:
top_altitud = (
    edges
    .sort_values(
        "elev_mean",
        ascending=False
    )
)


In [ ]:
print(
    top_altitud[
        ["name", "elev_mean"]
    ].head(20)
)

In [ ]:
steps = edges[
    edges["highway"] == "steps"
]

In [ ]:
edges["score"] = (
    edges["elev_mean"] * 0.5 +
    edges["desnivel"] * 2
)

In [ ]:
edges["score"] = (
    edges["elev_mean"] +
    5 * edges["pendiente_pct"]
)


In [ ]:
ranking = edges.sort_values(
    "score",
    ascending=False
)

In [ ]:
edges.to_file(
    "calles_altitud_madrid.gpkg",
    driver="GPKG"
)